In [ ]:
# 0. Cài đặt thư viện giải nén zstd (Bổ sung bắt buộc cho hệ thống Colab)
!apt-get update -qq
!apt-get install -y zstd pciutils lshw # Bổ sung thêm công cụ quét phần cứng GPU

# 1. Tự động tải và cài đặt Ollama Engine vào hệ thống Linux của Colab
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Khởi chạy dịch vụ Ollama Server ngầm dưới nền hệ thống
import subprocess
import time
import os

# Thiết lập biến môi trường bắt buộc
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_KEEP_ALIVE"] = "24h" # Giữ mô hình trong VRAM không bị unload

# Kiểm tra xem có tiến trình ollama nào đang chạy không, nếu có thì tắt đi để tránh lỗi port
subprocess.run(["pkill", "-f", "ollama serve"])
time.sleep(1)

# Khởi chạy Ollama ngầm, ghi log ra file thay vì DEVNULL để dễ debug nếu có lỗi
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("ollama.log", "w"),
    stderr=subprocess.STDOUT
)
print("Đang chờ Ollama Server khởi động...")
time.sleep(5) # Tăng thời gian chờ lên 5 giây để đảm bảo server khởi động hoàn toàn trên T4 GPU

# 3. Nạp trọng số mô hình Qwen2.5-7B-Instruct (Bản lượng hóa Int4 tối ưu tốc độ)
print("Đang tải trọng số mô hình Qwen2.5 (có thể mất 1-2 phút)...")
# Sử dụng subprocess thay vì ! để đồng bộ luồng thực thi trong Python
pull_result = subprocess.run(["ollama", "pull", "qwen2.5:7b-instruct"], capture_output=True, text=True)
if pull_result.returncode == 0:
    print("✅ Tải mô hình Qwen2.5 thành công!")
else:
    print(f"❌ Lỗi tải mô hình: {pull_result.stderr}")

# 4. Cài đặt gói công cụ điều phối mạng pyngrok
!pip install pyngrok -q
from pyngrok import ngrok

# ==============================================================================
# ĐIỀN THÔNG TIN XÁC THỰC MIỄN PHÍ CỦA BẠN VÀO ĐÂY
# ==============================================================================
NGROK_TOKEN = "3CIZ7D6081Xq1IpFd4Y7Lin6f0v_7aBumE5GDs3WpSfSJXHp1"
STATIC_DOMAIN = "animate-wobbling-rebuttal.ngrok-free.dev"

# Đảm bảo ngắt các kết nối cũ trước khi tạo kết nối mới
ngrok.kill()

ngrok.set_auth_token(NGROK_TOKEN)
try:
    # Thiết lập đường hầm cố định trỏ thẳng vào cổng dịch vụ 11434 của Ollama
    public_url = ngrok.connect(11434, proto="http", domain=STATIC_DOMAIN).public_url
    print("\n" + "="*70)
    print(f"🎉 KẾT NỐI HẠ TẦNG THÀNH CÔNG!")
    print(f"🔗 Ollama Endpoint URL vĩnh viễn của bạn là: {public_url}")
    print("="*70)
except Exception as e:
    print(f"❌ Lỗi thiết lập đường hầm cố định: {e}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 103 not upgraded.
Need to get 1,486 kB of archives.
After this operation, 4,951 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1 [322 kB]
Get:4 http://archive.ubuntu.com/ubuntu ja